# BÀI TẬP: E-COMMERCE DATA (ONLINE RETAIL)
**Nguồn:** kaggle.com/datasets/carrie1/ecommerce-data (541,909 dòng)


## Setup

In [1]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from scipy import stats

sns.set_style('whitegrid')

csv_path = 'https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv'

df = pd.read_csv(csv_path, encoding='ISO-8859-1', on_bad_lines='skip')
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print('Loaded from:', csv_path)
df.head()

Loaded from: https://raw.githubusercontent.com/databricks/Spark-The-Definitive-Guide/master/data/retail-data/all/online-retail-dataset.csv


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


---
# PHẦN A — DATA PROFILING
## A.1. Data size, column names, data types

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB


## A.2. Missing values & Duplicate data

In [ ]:
df.isnull().sum()
df.duplicated().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

## A.3. Invalid values

In [4]:
df.isna().sum()


InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

## A.4. Create a new column
Làm sạch dữ liệu (loại Quantity<=0, UnitPrice<=0), tạo cột `Sales` = Quantity * UnitPrice.

In [8]:
df = df.drop(df[df['Quantity']<=0].index)
df['Sales'] = df['Quantity'] * df['UnitPrice']

---
# PHẦN B — DESCRIPTIVE STATISTICS
## Group 1 — Central Tendency

In [9]:
print(df.mean(numeric_only=True))
print(df.median(numeric_only=True))

Quantity         10.655262
UnitPrice         3.857296
CustomerID    15294.315171
Sales            20.035500
dtype: float64
Quantity          3.00
UnitPrice         2.08
CustomerID    15159.00
Sales             9.90
dtype: float64


## Group 2 — Dispersion

In [11]:
num_df = df.select_dtypes('int', 'float')
range = num_df.max() - num_df.min()
var = num_df.var()
std = num_df.std()
IQR = num_df.quantile(0.75) - num_df.quantile(0.25)




## Group 3 — Location and Shape

In [12]:
Q1 = num_df.quantile(0.25)
Q2 = num_df.quantile(0.50)
Q3 = num_df.quantile(0.75)
skew = num_df.skew()
kurt = num_df.kurt()




In [13]:
df

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,Sales
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom,15.30
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom,22.00
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom,20.34
...,...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France,10.20
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France,12.60
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France,16.60
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France,16.60


---
# PHẦN C — DEFINE THE QUESTION

## Câu hỏi 1: Quốc gia nào đóng góp doanh thu cao nhất, chiếm bao nhiêu % tổng doanh thu?

In [32]:
dt_theo_qg = df.groupby('Country')['Sales'].sum()
print('Quốc gia có doanh thu cao nhất là: ', dt_theo_qg.idxmax(), 'với: ', dt_theo_qg.max())
print('chiếm ', (dt_theo_qg.max() / df['Sales'].sum() ) *100, '%')



Quốc gia có doanh thu cao nhất là:  United Kingdom với:  9003097.964
chiếm  84.57933071337507 %


## Câu hỏi 2: Sản phẩm nào bán chạy nhất theo doanh thu?

In [36]:
dt_theo_sp = df.groupby('StockCode')['Sales'].sum()
print('SP bán chạy nhất là', dt_theo_sp.idxmax(), 'với doanh thu: ', dt_theo_sp.max())



SP bán chạy nhất là DOT với doanh thu:  206248.77


## Câu hỏi 3: Doanh số có tính mùa vụ theo tháng không?

In [44]:
thang = df['InvoiceDate'].dt.month
dt_theo_thang = df.groupby(thang)['Sales'].sum()
print(dt_theo_thang)
print('Có vì tháng 9 - 12 có doanh thu cao đột biến so với các tháng còn lại')


InvoiceDate
1      691364.560
2      523631.890
3      717639.360
4      537808.621
5      770536.020
6      761739.900
7      719221.191
8      737014.260
9     1058590.172
10    1154979.300
11    1509496.330
12    1462538.820
Name: Sales, dtype: float64
Có vì tháng 9 - 12 có doanh thu cao đột biến so với các tháng còn lại


## Câu hỏi 4: Giá trị đơn hàng trung bình (Average Order Value) khác nhau thế nào giữa các quốc gia?

In [46]:
dg_giua_qg = df.groupby('Country')['UnitPrice'].mean()
print(dg_giua_qg)

Country
Australia                3.048523
Austria                  4.256030
Bahrain                  4.597778
Belgium                  3.630158
Brazil                   4.456250
Canada                   6.030331
Channel Islands          4.531618
Cyprus                   5.710391
Czech Republic           3.130800
Denmark                  3.146184
EIRE                     4.875849
European Community       4.830000
Finland                  5.296993
France                   4.399713
Germany                  3.708487
Greece                   4.574414
Hong Kong               23.474437
Iceland                  2.644011
Israel                   3.630441
Italy                    4.717955
Japan                    2.047383
Lebanon                  5.387556
Lithuania                2.841143
Malta                    4.867768
Netherlands              2.643982
Norway                   5.282155
Poland                   4.173364
Portugal                 5.843251
RSA                      4.277586
Saudi 

## Câu hỏi 5: Tỷ lệ giao dịch có dấu hiệu trả hàng/hủy (Quantity âm ở dữ liệu gốc) khác nhau thế nào giữa các quốc gia?

In [67]:
gd_qg = df.groupby('Country')['InvoiceNo'].count()
qg_am = df[df['Quantity'] < 0]
gd_qg_am = qg_am.groupby('Country')['InvoiceNo'].count()
ty_le_hang_bi_huy = (gd_qg_am / gd_qg) * 100
ty_le_hang_bi_huy = ty_le_hang_bi_huy.fillna(0).sort_values(ascending=False)
ty_le_hang_bi_huy



Country
Australia               0.0
Austria                 0.0
Bahrain                 0.0
Belgium                 0.0
Brazil                  0.0
Canada                  0.0
Channel Islands         0.0
Cyprus                  0.0
Czech Republic          0.0
Denmark                 0.0
EIRE                    0.0
European Community      0.0
Finland                 0.0
France                  0.0
Germany                 0.0
Greece                  0.0
Hong Kong               0.0
Iceland                 0.0
Israel                  0.0
Italy                   0.0
Japan                   0.0
Lebanon                 0.0
Lithuania               0.0
Malta                   0.0
Netherlands             0.0
Norway                  0.0
Poland                  0.0
Portugal                0.0
RSA                     0.0
Saudi Arabia            0.0
Singapore               0.0
Spain                   0.0
Sweden                  0.0
Switzerland             0.0
USA                     0.0
United Arab 

## Câu hỏi 6 (Tổng hợp) — Viết insight tổng hợp
Dựa trên Phần A, B, C, viết 4-5 câu insight tổng thể.

*(Viết insight của bạn vào đây...)*